# 📊 01 — Analyse Exploratoire des Données (EDA)
**Objectif** : Comprendre la structure des données, identifier les patterns liés au churn et détecter les anomalies avant la modélisation.

In [12]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv(r'C:\Users\imene\churn-prediction\data\WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f"Dimensions : {df.shape}")
print(f"Taux de churn : {(df['Churn']=='Yes').mean()*100:.1f}%")
df.head()

Dimensions : (7043, 21)
Taux de churn : 26.5%


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Vue générale du dataset

In [13]:
# Types, valeurs manquantes, statistiques
print("=== Types et valeurs manquantes ===")
missing = df.isnull().sum()
print(missing[missing > 0])

print("\n=== Statistiques descriptives ===")
df.describe()

=== Types et valeurs manquantes ===
TotalCharges    11
dtype: int64

=== Statistiques descriptives ===


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7032.000000
mean,0.162147,32.371149,64.761692,2283.300441
std,0.368612,24.559481,30.090047,2266.771362
min,0.000000,0.000000,18.250000,18.800000
25%,0.000000,9.000000,35.500000,401.450000
50%,0.000000,29.000000,70.350000,1397.475000
75%,0.000000,55.000000,89.850000,3794.737500
max,1.000000,72.000000,118.750000,8684.800000


## 2. Distribution du churn (variable cible)

In [14]:
churn_counts = df['Churn'].value_counts().reset_index()
churn_counts.columns = ['Churn', 'Count']
churn_counts['Pct'] = (churn_counts['Count'] / len(df) * 100).round(1)

fig = px.pie(
    churn_counts,
    names='Churn',
    values='Count',
    title='Distribution du churn',
    color='Churn',
    color_discrete_map={'Yes': '#EF553B', 'No': '#636EFA'},
    hole=0.4
)
fig.update_traces(textinfo='percent+label')
fig.show()

## 3. Variables numériques : tenure, MonthlyCharges, TotalCharges

In [25]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig = make_subplots(rows=1, cols=3, subplot_titles=num_cols)

for i, col in enumerate(num_cols, 1):
    for churn_val, color in [('No', "#2D30E7"), ('Yes', "#FF2600")]:
        subset = df[df['Churn'] == churn_val][col].dropna()
        fig.add_trace(
            go.Histogram(x=subset, name=f'Churn={churn_val}',
                         marker_color=color, opacity=0.6,
                         showlegend=(i == 1)),
            row=1, col=i
        )

fig.update_layout(
    title='Distribution des variables numériques par churn',
    barmode='overlay',
    height=400
)
fig.show()

## 4. Churn par variables catégorielles clés

In [16]:
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport', 'OnlineSecurity']

for col in cat_cols:
    churn_rate = (
        df.groupby(col)['Churn']
        .apply(lambda x: (x == 'Yes').mean() * 100)
        .reset_index()
    )
    churn_rate.columns = [col, 'ChurnRate']
    churn_rate = churn_rate.sort_values('ChurnRate', ascending=False)

    fig = px.bar(
        churn_rate,
        x=col, y='ChurnRate',
        title=f'Taux de churn par {col}',
        color='ChurnRate',
        color_continuous_scale='Reds',
        text=churn_rate['ChurnRate'].apply(lambda x: f'{x:.1f}%')
    )
    fig.update_traces(textposition='outside')
    fig.update_layout(yaxis_title='Taux de churn (%)', coloraxis_showscale=False)
    fig.show()

## 5. Ancienneté (tenure) vs Churn — boxplot

In [17]:
fig = px.box(
    df, x='Churn', y='tenure',
    color='Churn',
    color_discrete_map={'Yes': '#EF553B', 'No': '#636EFA'},
    title='Ancienneté des clients selon le churn',
    points='outliers'
)
fig.update_layout(yaxis_title='Ancienneté (mois)', showlegend=False)
fig.show()

## 6. MonthlyCharges vs tenure — scatter par churn

In [18]:
fig = px.scatter(
    df, x='tenure', y='MonthlyCharges',
    color='Churn',
    color_discrete_map={'Yes': '#EF553B', 'No': '#636EFA'},
    title='MonthlyCharges vs Ancienneté — coloré par churn',
    opacity=0.5,
    hover_data=['Contract', 'InternetService']
)
fig.show()

## 7. Matrice de corrélation (variables numériques)

In [27]:
df_corr = df[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
df_corr['Churn'] = (df['Churn'] == 'Yes').astype(int)

corr = df_corr.corr()

fig = px.imshow(
    corr,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    title='Matrice de corrélation',
    zmin=-1, zmax=1
)
fig.show()

## 8. Synthèse des insights EDA 

In [20]:
print("""
=== INSIGHTS CLÉS ===

1. DÉSÉQUILIBRE : 26.5% de churners → nécessite SMOTE ou class_weight
2. CONTRAT      : Les clients month-to-month churne beaucoup plus
3. ANCIENNETÉ   : Les churners sont majoritairement des clients récents (tenure faible)
4. CHARGES      : Les churners ont des MonthlyCharges plus élevées
5. SERVICES     : Absence de TechSupport et OnlineSecurity corrélée au churn
""")


=== INSIGHTS CLÉS ===

1. DÉSÉQUILIBRE : 26.5% de churners → nécessite SMOTE ou class_weight
2. CONTRAT      : Les clients month-to-month churne beaucoup plus
3. ANCIENNETÉ   : Les churners sont majoritairement des clients récents (tenure faible)
4. CHARGES      : Les churners ont des MonthlyCharges plus élevées
5. SERVICES     : Absence de TechSupport et OnlineSecurity corrélée au churn

